# RNAP Hack 2026 · Reto 2: News in 60 Seconds
**Equipo:** _Equipo 4_  ·  **Integrantes:** _Jaime Esquivel, Alberto Moreno, Jared Dávila, Randhu_

Reporte técnico (cuaderno principal). En la **Fase 1** se completan las secciones 1, 2 y 3.

In [ ]:
# Ejecuta esta celda una vez (Colab). Si ya tienes las librerías, no pasa nada.
!pip -q install transformers sentencepiece rouge_score accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Problema y datos

**Reto elegido:** News in 60 Seconds (Radio Cumbres).
**Problema:** resumir una noticia en español en **120 palabras o menos** (unos 60 segundos al aire, a ~2 palabras por segundo), usando solo hechos del artículo.
**Métrica principal:** ROUGE (ROUGE-1, ROUGE-2, ROUGE-L) contra el resumen del periodista, y % de resúmenes con ≤ 120 palabras.
**Fuente de los datos:** XL-Sum (csebuetnlp/xlsum), subconjunto en español: artículos de BBC Mundo con un resumen escrito por un periodista.

In [ ]:
import json, re, time, random
import numpy as np, pandas as pd
from pathlib import Path

# Carpeta de Google Drive donde dejaste los 3 archivos .jsonl (NO se suben a GitHub)
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/rnap_data')

train = pd.read_json(DATA_DIR/'spanish_train.jsonl', lines=True)
val   = pd.read_json(DATA_DIR/'spanish_val.jsonl',   lines=True)
test  = pd.read_json(DATA_DIR/'spanish_test.jsonl',  lines=True)

print('train:', len(train), '| val:', len(val), '| test:', len(test))
print('Columnas:', list(train.columns))

Mounted at /content/drive
train: 38110 | val: 4763 | test: 4763
Columnas: ['id', 'url', 'title', 'summary', 'text']


In [ ]:
# Tamaño de cada partición y longitud típica (palabras)
resumen = pd.DataFrame({
    'artículos':        [len(train), len(val), len(test)],
    'palabras texto (mediana)':    [train.text.str.split().str.len().median(),
                                    val.text.str.split().str.len().median(),
                                    test.text.str.split().str.len().median()],
    'palabras resumen (mediana)':  [train.summary.str.split().str.len().median(),
                                    val.summary.str.split().str.len().median(),
                                    test.summary.str.split().str.len().median()],
}, index=['train', 'val', 'test'])
resumen

,artículos,palabras texto (mediana),palabras resumen (mediana)
train,38110,771.0,27.0
val,4763,769.0,27.0
test,4763,768.0,27.0


**División de datos.** XL-Sum ya trae la división en entrenamiento (38,110), validación (4,763) y prueba (4,763), hecha **antes de entrenar**. Reglas del equipo:
- *train*: ajustar el modelo base (vocabulario e IDF de TF-IDF).
- *val*: elegir parámetros del modelo base (longitud objetivo del resumen).
- *test*: **solo** para las métricas finales, una vez.

Por tiempo de cómputo, ambos modelos se evalúan sobre **la misma muestra aleatoria fija** del conjunto de prueba (semilla 42).

In [ ]:
SEED = 42
N_EVAL = 200   # sube a 500 si Colab alcanza

test_s = test.sample(N_EVAL, random_state=SEED).reset_index(drop=True)
val_s  = val.sample(200,    random_state=SEED).reset_index(drop=True)
print(len(test_s), 'artículos de prueba |', len(val_s), 'de validación')

200 artículos de prueba | 200 de validación


## 2. Representación del texto

**Modelo base: TF-IDF por oración.** Cada oración se representa como un vector TF-IDF y su puntaje es el peso promedio de sus palabras. TF-IDF da peso alto a palabras frecuentes en la noticia pero poco comunes en el corpus (nombres, cifras, temas) y peso bajo a palabras vacías ("de", "la", "que"). Así una oración con muchas palabras informativas puntúa más. Es una representación de las primeras etapas del curso: no necesita GPU, es rápida y es fácil de explicar.

**Modelo preentrenado (sin ajuste): mT5_multilingual_XLSum.** Usa el tokenizador SentencePiece del propio modelo (subpalabras) y embeddings aprendidos; se aplica tal como viene.

## 3. Modelo base y preentrenado sin ajuste

### 3.1 Métrica (ROUGE para español)
El tokenizador por defecto de `rouge_score` borra las letras con acento, así que usamos uno propio que conserva `á é í ó ú ñ`.

In [ ]:
from rouge_score import rouge_scorer

class EsTokenizer:
    def tokenize(self, text):
        return re.findall(r'\w+', text.lower())

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], tokenizer=EsTokenizer())

def rouge_eval(preds, refs):
    acc = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for k in acc:
            acc[k].append(s[k].fmeasure)
    return {k: round(100*float(np.mean(v)), 2) for k, v in acc.items()}

def pct_within(preds, limit=120):
    return round(100*float(np.mean([len(p.split()) <= limit for p in preds])), 1)

### 3.2 Modelo base: resumen extractivo con TF-IDF
Se ajusta el vocabulario y el IDF con una muestra de *train*. Después se puntúa cada oración y se eligen las mejores sin pasar del límite de palabras.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

t0 = time.time()
tfidf = TfidfVectorizer(lowercase=True, max_features=50000, sublinear_tf=True, norm=None)
tfidf.fit(train['text'].sample(20000, random_state=SEED))   # "entrenamiento" del modelo base
base_fit_time = time.time() - t0
print(f'TF-IDF ajustado en {base_fit_time:.1f} s | vocabulario: {len(tfidf.vocabulary_)} palabras')

def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sents if len(s.split()) >= 4]

def extractive_summary(text, max_words=120):
    sents = split_sentences(text)
    if not sents:
        return ' '.join(text.split()[:max_words])
    X = tfidf.transform(sents)
    lengths = np.array([len(s.split()) for s in sents])
    scores = np.asarray(X.sum(axis=1)).ravel() / np.maximum(lengths, 1)   # peso TF-IDF promedio
    chosen, total = [], 0
    for i in np.argsort(-scores):
        if total + lengths[i] <= max_words:
            chosen.append(i)
            total += lengths[i]
    if not chosen:                                   # primera oración demasiado larga
        return ' '.join(sents[int(np.argmax(scores))].split()[:max_words])
    return ' '.join(sents[i] for i in sorted(chosen))   # en el orden original

# Ejemplo
ej = test_s.loc[0, 'text']
print(extractive_summary(ej))

TF-IDF ajustado en 21.2 s | vocabulario: 50000 palabras
En esta fecha declarada feriado, se sientan a la mesa para celebrar el Día de Acción de Gracias (ThanksgivingDay), compartiendo pavo horneado, salsa de arándanos, puré de manzana y pastel de calabaza, entre otros platos. La costumbre hace que los que comparten la comida digan en alto de qué están agradecidos. El presidente Abraham Lincoln oficializó la celebración de Acción de Gracias en 1863. El gobernante ordenó celebrar Acción de Gracias luego de una campaña de la poeta y editora Sarah Josepha Hale, que había leído sobre el banquete de los colonizadores y los nativos de 1621, según cuenta la Enciclopedia Británica. Desde entonces, casi todo EE.UU. celebra Acción de Gracias. ¿Ya conoces nuestro canal de YouTube?


**Elegir la longitud objetivo con validación** (nunca con *test*). Todas las opciones respetan el límite de 120 palabras.

In [ ]:
resultados_val = {}
for L in [30, 45, 60, 90, 120]:
    preds = [extractive_summary(t, max_words=L) for t in val_s['text']]
    resultados_val[L] = rouge_eval(preds, val_s['summary'])['rouge1']
print('ROUGE-1 en validación por longitud objetivo:', resultados_val)

BEST_L = max(resultados_val, key=resultados_val.get)
print('Longitud elegida para el modelo base:', BEST_L, 'palabras')

ROUGE-1 en validación por longitud objetivo: {30: 17.45, 45: 18.83, 60: 19.08, 90: 17.76, 120: 16.47}
Longitud elegida para el modelo base: 60 palabras


In [ ]:
# Resultados del modelo base sobre el conjunto de PRUEBA (mide también el tiempo)
t0 = time.time()
base_preds = [extractive_summary(t, max_words=BEST_L) for t in test_s['text']]
base_time = time.time() - t0

base_res = rouge_eval(base_preds, test_s['summary'])
base_res['% ≤120 palabras'] = pct_within(base_preds)
base_res['tiempo total (s)'] = round(base_time, 2)
base_res['s por artículo']   = round(base_time/len(test_s), 3)
print(base_res)

{'rouge1': 18.78, 'rouge2': 2.82, 'rougeL': 12.2, '% ≤120 palabras': 100.0, 'tiempo total (s)': 0.32, 's por artículo': 0.002}


### 3.3 Preentrenado sin ajuste: mT5_multilingual_XLSum
Modelo encoder-decoder (Transformer) ya entrenado para resumir en varios idiomas, incluido el español. Se usa **tal como viene**, sin entrenarlo más, con los parámetros de generación de su documentación. *En Colab activa GPU:* Entorno de ejecución › Cambiar tipo › GPU T4.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = 'csebuetnlp/mT5_multilingual_XLSum'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', device)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device).eval()

clean = lambda x: re.sub(r'\s+', ' ', x.strip())

def mt5_summaries(texts, bs=8):
    out = []
    for i in range(0, len(texts), bs):
        batch = [clean(t) for t in texts[i:i+bs]]
        enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            ids = model.generate(**enc, max_length=84, num_beams=4, no_repeat_ngram_size=2)
        out += tok.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return out

t0 = time.time()
pre_preds = mt5_summaries(list(test_s['text']))
pre_time = time.time() - t0

pre_res = rouge_eval(pre_preds, test_s['summary'])
pre_res['% ≤120 palabras'] = pct_within(pre_preds)
pre_res['tiempo total (s)'] = round(pre_time, 2)
pre_res['s por artículo']   = round(pre_time/len(test_s), 3)
print(pre_res)

Dispositivo: cuda


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.33GB            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model.safetensors: downloading bytes:           |  0.00B            

{'rouge1': 30.11, 'rouge2': 11.34, 'rougeL': 22.98, '% ≤120 palabras': 100.0, 'tiempo total (s)': 83.68, 's por artículo': 0.418}


### 3.4 Tabla comparativa

In [ ]:
tabla = pd.DataFrame({
    f'Base: TF-IDF extractivo ({BEST_L} palabras)': base_res,
    'Preentrenado sin ajuste: mT5_multilingual_XLSum': pre_res,
}).T
tabla

,rouge1,rouge2,rougeL,% ≤120 palabras,tiempo total (s),s por artículo
Base: TF-IDF extractivo (60 palabras),18.78,2.82,12.20,100.0,0.32,0.002
Preentrenado sin ajuste: mT5_multilingual_XLSum,30.11,11.34,22.98,100.0,83.68,0.418


In [ ]:
# Un ejemplo lado a lado (útil para la conclusión y para Fase 2)
i = 3
print('REFERENCIA (periodista):\n', test_s.loc[i, 'summary'], '\n')
print('BASE TF-IDF:\n', base_preds[i], '\n')
print('mT5 sin ajuste:\n', pre_preds[i])

REFERENCIA (periodista):
 Un plato de arroz con pollo y salsa soya que se vende por menos de US$2 es la especialidad de Hong Kong Soya Sauce Chicken Rice and Noodle, un puesto ambulante en las calles de Singapur que acaba de recibir una estrella Michelin. 

BASE TF-IDF:
 La concurrencia al puesto de comida quedó en evidencia cuando nuestra entrevista telefónica estuvo varias veces interrumpida mientras Chan Hon Meng preparaba los pedidos. La Guía Michelin Singapur menciona 62 puestos de comida callejeros, incluyendo Hong Kong Soya Sauce Chicken Rice and Noodle con su distintiva estrella a la derecha. 

mT5 sin ajuste:
 El dueño de un puesto ambulante de comida en Hong Kong dijo a BBC Mundo que todavía no ha decidido dónde colgar una estrella Michelin.


### 3.5 Conclusión (Fase 1)

Ambos modelos se evaluaron sobre la misma muestra de 200 artículos del conjunto de prueba (semilla 42). La longitud del modelo base (60 palabras) se eligió con validación, no con prueba.

| Modelo | ROUGE-1 | ROUGE-2 | ROUGE-L | % ≤ 120 palabras | Tiempo total | s por artículo |
| --- | --- | --- | --- | --- | --- | --- |
| Base: TF-IDF extractivo (60 palabras) | 18.78 | 2.82 | 12.20 | 100 % | 0.32 s | 0.002 |
| mT5_multilingual_XLSum sin ajuste | 30.11 | 11.34 | 22.98 | 100 % | 83.68 s (GPU) | 0.418 |

- **Calidad:** el preentrenado supera al base en las tres métricas. En ROUGE-1 la diferencia es de 11.3 puntos (30.11 contra 18.78) y en ROUGE-2 es de casi cuatro veces (11.34 contra 2.82). Esto indica que el modelo base casi no coincide con el resumen del periodista en secuencias de dos palabras.
- **Velocidad:** el base es mucho más rápido, unas 260 veces (0.32 s contra 83.68 s para los 200 artículos). Aun así, 0.42 s por artículo con el mT5 en GPU es aceptable para un uso en la radio.
- **Límite de 120 palabras:** los dos modelos lo cumplen en el 100 % de los casos. El base lo garantiza por construcción y el mT5 porque genera como máximo 84 tokens.
- **Fallas del base:** elige oraciones sueltas del artículo y no las conecta. En el ejemplo de la sección 3.4 toma una oración sobre la entrevista telefónica y no menciona el platillo ni su precio, que es lo importante del texto.
- **Fallas del preentrenado:** puede cambiar un hecho al reescribirlo. En ese mismo ejemplo dice que el dueño todavía no decide dónde colgar la estrella Michelin, cuando el artículo dice que el puesto la recibió. Esto es justo el problema de datos inventados que preocupa a Radio Cumbres. Además, el mT5 solo lee los primeros 512 tokens y las noticias tienen una mediana de unas 770 palabras, por lo que es probable que no vea el final del artículo.
- **Siguiente paso (Fase 2):** ajustar el mT5 con los datos de entrenamiento, evaluar en prueba con las mismas métricas y agregar la revisión de cifras y nombres que no aparecen en el artículo para detectar datos inventados.

---